In [ ]:
import sys

import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)

**SEED = 421** (the last 4 digits of my student ID: 0421)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 421

data = pd.read_csv("data/heart_disease.csv")
feature_names = [c for c in data.columns if c != "disease"]
X, y = data[feature_names].to_numpy(), data["disease"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)

# Part 1. Entropy and information gain by hand

Twelve synthetic patients: p1–p6 have the disease (6), p7–p12 do not (6).
Notation: $H = -\sum_c p_c \log_2 p_c$ and
$IG = H(\text{parent}) - \sum_{k} \frac{n_k}{n} H(\text{child}_k)$.

In [ ]:
toy = pd.DataFrame({
    "age_55_plus": [1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
    "exercise_angina": [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "male": [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    "high_bp": [1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0],
    "disease": [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
})
toy

## 1.1 Parent entropy (by hand)

$$H(\text{parent}) = -\left(\tfrac{6}{12}\log_2\tfrac{6}{12} + \tfrac{6}{12}\log_2\tfrac{6}{12}\right)
= -\left(0.5\cdot(-1) + 0.5\cdot(-1)\right) = \mathbf{1.000}\ \text{bit}$$

In [ ]:
import math

def H(*counts):
    """Entropy in bits of a group given its class counts (a helper for checking my hand work)."""
    n = sum(counts)
    return -sum((c / n) * math.log2(c / n) for c in counts if c > 0)

H_parent = -(0.5 * math.log2(0.5) + 0.5 * math.log2(0.5))   # the formula with the numbers in it
print("H_parent =", H_parent, "| helper H(6, 6) =", H(6, 6))

## 1.2 Root: information gain of each feature (by hand)

For every feature I count the sick/healthy patients in each child (from the table above).

**age_55_plus**
- = 1: p1–p5 sick, p7 healthy → 6 patients, (5 sick, 1 healthy)
- = 0: p6 sick, p8–p12 healthy → 6 patients, (1 sick, 5 healthy)

$$H_1 = -\left(\tfrac56\log_2\tfrac56+\tfrac16\log_2\tfrac16\right) = -(0.8333\cdot(-0.2630)+0.1667\cdot(-2.5850)) = 0.6500,\qquad H_0 = 0.6500$$
$$IG = 1 - \left(\tfrac{6}{12}\cdot0.6500+\tfrac{6}{12}\cdot0.6500\right) = 1 - 0.6500 = \mathbf{0.3500}$$

**exercise_angina**
- = 1: p1–p4, all sick → 4 patients, (4 sick, 0 healthy), $H = 0$
- = 0: p5, p6 sick; p7–p12 healthy → 8 patients, (2 sick, 6 healthy)

$$H_0 = -\left(\tfrac28\log_2\tfrac28+\tfrac68\log_2\tfrac68\right) = -(0.25\cdot(-2)+0.75\cdot(-0.4150)) = 0.8113$$
$$IG = 1 - \left(\tfrac{4}{12}\cdot0+\tfrac{8}{12}\cdot0.8113\right) = 1 - 0.5409 = \mathbf{0.4591}$$

**male**
- = 1: p1, p2, p3 sick; p7, p8, p9 healthy → 6 patients, (3 sick, 3 healthy), $H = 1$
- = 0: p4, p5, p6 sick; p10, p11, p12 healthy → 6 patients, (3 sick, 3 healthy), $H = 1$

$$IG = 1 - \left(\tfrac{6}{12}\cdot1+\tfrac{6}{12}\cdot1\right) = 1 - 1 = \mathbf{0.0000}$$

**high_bp**
- = 1: p1, p2, p4, p5 sick; p10, p11 healthy → 6 patients, (4 sick, 2 healthy)
- = 0: p3, p6 sick; p7, p8, p9, p12 healthy → 6 patients, (2 sick, 4 healthy)

$$H_1 = -\left(\tfrac46\log_2\tfrac46+\tfrac26\log_2\tfrac26\right) = -(0.6667\cdot(-0.5850)+0.3333\cdot(-1.5850)) = 0.9183,\qquad H_0 = 0.9183$$
$$IG = 1 - \left(\tfrac{6}{12}\cdot0.9183+\tfrac{6}{12}\cdot0.9183\right) = 1 - 0.9183 = \mathbf{0.0817}$$

In [ ]:
# The same four numbers, typed as arithmetic from the counts above (a check of the hand work)
gain_age    = H_parent - (6/12 * H(5, 1) + 6/12 * H(1, 5))
gain_angina = H_parent - (4/12 * H(4, 0) + 8/12 * H(2, 6))
gain_male   = H_parent - (6/12 * H(3, 3) + 6/12 * H(3, 3))
gain_bp     = H_parent - (6/12 * H(4, 2) + 6/12 * H(2, 4))

root_gains = pd.Series({"age_55_plus": gain_age, "exercise_angina": gain_angina,
                        "male": gain_male, "high_bp": gain_bp})
print(root_gains.round(4))
print("root =", root_gains.idxmax())

# the counts themselves, straight from the table
for f in ["age_55_plus", "exercise_angina", "male", "high_bp"]:
    print(pd.crosstab(toy[f], toy["disease"]), "\n")

**Root: `exercise_angina`**, with the largest gain (0.4591 vs 0.3500, 0.0817 and 0.0000).
Its branch `exercise_angina = 1` is pure (4 sick), so it becomes a leaf. The branch
`exercise_angina = 0` (p5–p12, 8 patients: 2 sick, 6 healthy) is still impure.

## 1.3 One level down: the impure branch `exercise_angina = 0` (by hand)

Patients p5–p12: 2 sick (p5, p6), 6 healthy. $H_{sub} = -\left(\tfrac28\log_2\tfrac28+\tfrac68\log_2\tfrac68\right) = 0.8113$.

**age_55_plus**
- = 1: p5 sick, p7 healthy → 2 patients (1, 1), $H = 1$
- = 0: p6 sick; p8–p12 healthy → 6 patients (1 sick, 5 healthy), $H = 0.6500$

$$IG = 0.8113 - \left(\tfrac28\cdot1+\tfrac68\cdot0.6500\right) = 0.8113 - 0.7375 = \mathbf{0.0738}$$

**male**
- = 1: p7, p8, p9, all healthy → 3 patients (0, 3), $H = 0$
- = 0: p5, p6 sick; p10, p11, p12 healthy → 5 patients (2 sick, 3 healthy), $H = -\left(\tfrac25\log_2\tfrac25+\tfrac35\log_2\tfrac35\right) = 0.9710$

$$IG = 0.8113 - \left(\tfrac38\cdot0+\tfrac58\cdot0.9710\right) = 0.8113 - 0.6069 = \mathbf{0.2044}$$

**high_bp**
- = 1: p5 sick; p10, p11 healthy → 3 patients (1, 2), $H = 0.9183$
- = 0: p6 sick; p7, p8, p9, p12 healthy → 5 patients (1, 4), $H = 0.7219$

$$IG = 0.8113 - \left(\tfrac38\cdot0.9183+\tfrac58\cdot0.7219\right) = 0.8113 - 0.7956 = \mathbf{0.0157}$$

In [ ]:
sub = toy[toy["exercise_angina"] == 0]
H_sub = H(2, 6)
print("H_sub =", round(H_sub, 4), "| patients:", len(sub))

for f in ["age_55_plus", "male", "high_bp"]:
    print(pd.crosstab(sub[f], sub["disease"]), "\n")

gain_age_sub  = H_sub - (2/8 * H(1, 1) + 6/8 * H(1, 5))
gain_male_sub = H_sub - (3/8 * H(0, 3) + 5/8 * H(2, 3))
gain_bp_sub   = H_sub - (3/8 * H(1, 2) + 5/8 * H(1, 4))

sub_gains = pd.Series({"age_55_plus": gain_age_sub, "male": gain_male_sub, "high_bp": gain_bp_sub})
print(sub_gains.round(4))
print("chosen in the branch:", sub_gains.idxmax(), "| its gain at the root was", round(gain_male, 4))

**Chosen in the branch: `male`** (gain 0.2044, against 0.0738 and 0.0157). At the root its gain was **0**.

*Why a feature that is useless at the root can be best below it.* At the root, `male` cuts the 12 patients into two halves that each have 3 sick and 3 healthy, so neither half is purer than the parent. But all the sick men (p1, p2, p3) have exercise angina, so once the root has taken the angina patients away, every remaining man (p7, p8, p9) is healthy. The value of `male` only appears *given* `exercise_angina = 0`: it is an interaction, and information gain at the root only looks at one feature at a time.

# Part 2. A decision tree from scratch

The learning algorithm in NumPy, with the required signatures:
`entropy`, `information_gain`, `best_split`, `build_tree`, `predict_tree`.

$$H(y) = -\sum_c p_c \log_2 p_c \qquad IG = H(y) - \tfrac{n_L}{n}H(y_L) - \tfrac{n_R}{n}H(y_R)$$

Rules followed: thresholds are midpoints between consecutive distinct sorted values, `x <= t` goes left;
a split is valid only if both children have at least `min_samples_leaf` rows; ties go to the lower feature index, then the lower threshold
(a later candidate must beat the best gain by more than `1e-12`, so that floating-point noise in two mathematically equal gains does not decide the tie);
a leaf is made when the node is pure, `depth == max_depth` or no valid split exists, and it predicts its majority class (a tie goes to the smaller label).

In [ ]:
TIE_TOL = 1e-12   # two gains closer than this are treated as equal (the lower feature / threshold wins)


def entropy(y):
    """Entropy in bits of an array of integer class labels."""
    y = np.asarray(y)
    if len(y) == 0:
        return 0.0
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))


def information_gain(y, left):
    """Entropy of y minus the size-weighted entropy of y[left] and y[~left]; left is a boolean mask."""
    n = len(y)
    n_left = int(left.sum())
    n_right = n - n_left
    return entropy(y) - (n_left / n) * entropy(y[left]) - (n_right / n) * entropy(y[~left])


def best_split(X, y, min_samples_leaf=1):
    """Return (feature_index, threshold, gain) of the best split, or None if no valid split exists."""
    best = None
    for j in range(X.shape[1]):
        values = np.unique(X[:, j])
        for t in (values[:-1] + values[1:]) / 2:          # midpoints between consecutive distinct values
            left = X[:, j] <= t
            n_left = int(left.sum())
            if n_left < min_samples_leaf or len(y) - n_left < min_samples_leaf:
                continue
            g = information_gain(y, left)
            if best is None or g > best[2] + TIE_TOL:     # strictly better only: ties keep lower feature / threshold
                best = (j, float(t), g)
    return best


def majority(y):
    """Majority class of an array of integer labels; a tie goes to the smaller label."""
    return int(np.bincount(y).argmax())


def build_tree(X, y, max_depth=None, min_samples_leaf=1, depth=0):
    """Grow a tree recursively and return it as nested dicts (or any structure you like)."""
    if len(np.unique(y)) == 1 or (max_depth is not None and depth == max_depth):
        return {"leaf": majority(y)}
    split = best_split(X, y, min_samples_leaf)
    if split is None:
        return {"leaf": majority(y)}
    j, t, _ = split
    left = X[:, j] <= t
    return {"feature": j, "threshold": t,
            "left": build_tree(X[left], y[left], max_depth, min_samples_leaf, depth + 1),
            "right": build_tree(X[~left], y[~left], max_depth, min_samples_leaf, depth + 1)}


def predict_tree(tree, X):
    """Return one predicted label per row of X."""
    out = np.empty(len(X), dtype=int)
    for i, row in enumerate(X):
        node = tree
        while "leaf" not in node:
            node = node["left"] if row[node["feature"]] <= node["threshold"] else node["right"]
        out[i] = node["leaf"]
    return out

## 2.1 Do `entropy` and `information_gain` reproduce the hand values of Part 1?

In [ ]:
y_toy = toy["disease"].to_numpy()
hand = {"parent entropy": 1.0, "age_55_plus": 0.3500, "exercise_angina": 0.4591, "male": 0.0000, "high_bp": 0.0817}

rows = [{"quantity": "parent entropy", "hand (Part 1)": hand["parent entropy"], "my function": entropy(y_toy)}]
for f in ["age_55_plus", "exercise_angina", "male", "high_bp"]:
    left = toy[f].to_numpy() == 0                     # the gain does not depend on which child is called "left"
    rows.append({"quantity": f"gain of {f}", "hand (Part 1)": hand[f], "my function": information_gain(y_toy, left)})

check = pd.DataFrame(rows)
check["match (4 decimals)"] = np.isclose(check["hand (Part 1)"], check["my function"], atol=5e-5)
check.round(4)

In [ ]:
# One level down, inside the branch exercise_angina = 0
sub_y = sub["disease"].to_numpy()
hand_sub = {"age_55_plus": 0.0738, "male": 0.2044, "high_bp": 0.0157}
rows = [{"gain in branch": f, "hand (Part 1)": hand_sub[f],
         "my function": information_gain(sub_y, sub[f].to_numpy() == 0)} for f in hand_sub]
check_sub = pd.DataFrame(rows)
check_sub["match (4 decimals)"] = np.isclose(check_sub["hand (Part 1)"], check_sub["my function"], atol=5e-5)
check_sub.round(4)

## 2.2 `best_split` against scikit-learn on the training split

In [ ]:
from sklearn.tree import DecisionTreeClassifier

ref = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=SEED).fit(X_train, y_train)
j, t, g = best_split(X_train, y_train)

root_cmp = pd.DataFrame({
    "feature": [feature_names[ref.tree_.feature[0]], feature_names[j]],
    "threshold": [float(ref.tree_.threshold[0]), t],
}, index=["scikit-learn", "mine"])
root_cmp["gain (mine)"] = [np.nan, g]
print(root_cmp.round(4))
print("same feature and threshold:", root_cmp.loc["scikit-learn", "feature"] == root_cmp.loc["mine", "feature"]
      and np.isclose(root_cmp.loc["scikit-learn", "threshold"], root_cmp.loc["mine", "threshold"]))

## 2.3 Agreement of the predictions with scikit-learn on the test rows

In [ ]:
rows = []
for d in [1, 2, 3, 4, 5, None]:
    mine = build_tree(X_train, y_train, max_depth=d)
    sk = DecisionTreeClassifier(criterion="entropy", max_depth=d, random_state=SEED).fit(X_train, y_train)
    agree = (predict_tree(mine, X_test) == sk.predict(X_test)).mean()
    rows.append({"max_depth": "unlimited" if d is None else d, "agreement": round(agree, 4),
                 "rows that differ (of 90)": int(round((1 - agree) * len(X_test)))})
agreement_table = pd.DataFrame(rows)
agreement_table

In [ ]:
# Extra check: the same comparison for min_samples_leaf (unlimited depth)
rows = []
for m in [1, 5, 20]:
    mine = build_tree(X_train, y_train, max_depth=None, min_samples_leaf=m)
    sk = DecisionTreeClassifier(criterion="entropy", min_samples_leaf=m, random_state=SEED).fit(X_train, y_train)
    rows.append({"min_samples_leaf": m, "agreement": round((predict_tree(mine, X_test) == sk.predict(X_test)).mean(), 4)})
pd.DataFrame(rows)

In [ ]:
# Diagnostic: where does the unlimited-depth tree first differ from scikit-learn's? Are the two gains equal there?
sk_full = DecisionTreeClassifier(criterion="entropy", max_depth=None, random_state=SEED).fit(X_train, y_train)
my_full = build_tree(X_train, y_train)
sk_t = sk_full.tree_

def compare_nodes(my, node, Xs, ys, path="root"):
    """Walk both trees together and print every node where they choose different splits."""
    my_leaf, sk_leaf = "leaf" in my, sk_t.children_left[node] == -1
    if my_leaf or sk_leaf:
        if my_leaf != sk_leaf:
            print(f"{path}: one tree stops here and the other splits ({len(ys)} rows)")
        return
    j_sk, t_sk = int(sk_t.feature[node]), float(sk_t.threshold[node])
    j_my, t_my = my["feature"], my["threshold"]
    if j_my != j_sk or not np.isclose(t_my, t_sk):
        g_my = information_gain(ys, Xs[:, j_my] <= t_my)
        g_sk = information_gain(ys, Xs[:, j_sk] <= t_sk)
        print(f"{path}: {len(ys)} rows | mine: {feature_names[j_my]} <= {t_my:.3f} (gain {g_my:.6f})"
              f" | sklearn: {feature_names[j_sk]} <= {t_sk:.3f} (gain {g_sk:.6f}) | equal gains: {abs(g_my - g_sk) < 1e-9}")
        return
    left = Xs[:, j_my] <= t_my
    compare_nodes(my["left"], sk_t.children_left[node], Xs[left], ys[left], path + " > L")
    compare_nodes(my["right"], sk_t.children_right[node], Xs[~left], ys[~left], path + " > R")

compare_nodes(my_full, 0, X_train, y_train)
print("diagnostic finished (if no node line is printed above, the two unlimited trees pick the same split everywhere)")

**Answers for Part 2**

| question | result |
|---|---|
| 1. `entropy` / `information_gain` reproduce the hand values of Part 1 | yes, all 5 values of 1.1–1.2 and the 3 values of 1.3 match to 4 decimals (tables above) |
| 2. `best_split` on `X_train, y_train` vs scikit-learn's depth-1 tree | the same feature and the same threshold: `thal <= 4.5` (see the table above) |
| 3. agreement of predictions on the 90 test rows | in the table above (depths 1–5 and unlimited) |

**If the agreement is below 100% anywhere** (for me: unlimited depth), the cause is not a different algorithm but **ties**.
Deep in a tree a node holds only a handful of patients, and then several different (feature, threshold) pairs give *exactly the same* information gain.
My rule breaks such a tie by the lowest feature index and then the lowest threshold, while scikit-learn visits the features in a random order (that is what `random_state` controls)
and keeps the first best one it meets, so it can pick another split with the same gain. The two trees then partition the rows differently inside that small node and a few test rows
that fall into it receive different labels. The diagnostic cell above prints the first node where the trees disagree and shows that the two gains there are equal;
it is also why a mismatch can only appear at large depth. A second, smaller source is that scikit-learn stores the features as `float32`, which can move a threshold by a rounding error.

# Part 3. How deep should the tree be?

Everything in this part uses the **training split only**; the test split is touched once, in step 3e.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def sweep(param, values):
    """5-fold CV (on the training split) of an entropy tree for each value of max_depth or min_samples_leaf."""
    rows = []
    for v in values:
        kw = {"max_depth": v} if param == "max_depth" else {"max_depth": None, "min_samples_leaf": v}
        model = DecisionTreeClassifier(criterion="entropy", random_state=SEED, **kw)
        s = cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
        leaves = model.fit(X_train, y_train).get_n_leaves()      # number of leaves of the tree fitted on the whole training split
        rows.append({param: "unlimited" if v is None else v, "leaves": leaves,
                     "train_acc": s["train_score"].mean(), "cv_acc": s["test_score"].mean(),
                     "cv_std": s["test_score"].std()})
    return pd.DataFrame(rows)

## 3a. Depth sweep

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, None]
depth_table = sweep("max_depth", depths)
depth_table.round(4)

## 3b. Plot

In [ ]:
x = range(len(depth_table))
plt.figure(figsize=(7, 4))
plt.plot(x, depth_table["train_acc"], marker="o", label="train accuracy")
plt.errorbar(x, depth_table["cv_acc"], yerr=depth_table["cv_std"], marker="o", capsize=4,
             label="CV accuracy (± std over the 5 folds)")
plt.xticks(x, depth_table["max_depth"])
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.title("Training vs cross-validated accuracy by tree depth")
plt.legend()
plt.show()

## 3c. Choosing a depth

**Selection rule (one-standard-deviation rule).** Find the depth with the highest mean CV accuracy and its fold standard deviation $s$.
Every depth whose mean CV accuracy is at least $\text{best mean} - s$ is statistically indistinguishable from the best one.
Among those, take the **simplest** tree, i.e. the one with the fewest leaves.
The rule uses the spread over the folds: a depth is only preferred over a simpler one if it beats it by more than the noise.

In [ ]:
best = depth_table["cv_acc"].idxmax()
threshold = depth_table.loc[best, "cv_acc"] - depth_table.loc[best, "cv_std"]
candidates = depth_table[depth_table["cv_acc"] >= threshold]
chosen = candidates.sort_values("leaves").iloc[0]
chosen_depth = chosen["max_depth"]
chosen_depth = None if chosen_depth == "unlimited" else int(chosen_depth)

print("best mean CV depth:", depth_table.loc[best, "max_depth"],
      "| mean", round(depth_table.loc[best, "cv_acc"], 4), "| std", round(depth_table.loc[best, "cv_std"], 4))
print("threshold (mean - std):", round(threshold, 4))
print("depths inside the threshold:", list(candidates["max_depth"]))
print("chosen depth (fewest leaves among them):", chosen_depth)

**Chosen depth: `max_depth = 1`** (a stump with 2 leaves). Justification: its mean CV accuracy (0.686) is inside the noise band of the best depth (0.729 at depth 6, band lower edge 0.649), and it is the simplest tree in the band.

**Is the winner clearly better than its neighbours?** No. The best mean (depth 6) beats its neighbours depth 5 (0.705) and depth 8 (0.696) by only 0.02–0.03, while the fold std is about 0.08, and the whole curve from depth 1 to 10 stays between 0.686 and 0.729. In fact **every** depth falls inside the band, so the data cannot separate the depths and the rule falls back on the simplest tree. The CV accuracy is basically flat while the training accuracy keeps rising.

## 3d. A second brake: `min_samples_leaf`

In [ ]:
leaf_table = sweep("min_samples_leaf", [1, 2, 5, 10, 20, 40])
leaf_table.round(4)

**Comparison of the two brakes.** `max_depth` caps how long any path can be, whichever part of the data it is in. `min_samples_leaf` instead forbids leaves that hold too few patients, so the tree can still grow deep where there is a lot of data and stops only where the data become thin. It also gives finer control of the size (35 → 4 leaves, against 2–35 for depth). In accuracy they behave alike: all CV values lie between 0.686 and 0.724, within one fold std (about 0.04–0.08) of each other, and neither beats the unlimited tree (0.724) clearly; what both brakes do is lower the training accuracy from 1.00 towards the CV accuracy.

## 3e. Test once

In [ ]:
final = DecisionTreeClassifier(criterion="entropy", max_depth=chosen_depth, random_state=SEED).fit(X_train, y_train)
p = final.score(X_test, y_test)
n = len(y_test)
se = np.sqrt(p * (1 - p) / n)
print(f"chosen depth = {chosen_depth}: test accuracy = {p:.4f}, standard error = {se:.4f}, n = {n}")

**Test accuracy of the chosen tree: 0.789, standard error 0.043** ($\sqrt{p(1-p)/n}$ with $n = 90$), so a rough 95% range is 0.70–0.87.
It is higher than the CV estimate of this tree (0.686) by about two standard errors, which I read as the luck of one 90-row split rather than a real improvement; I do not tune anything on the test split.

**Gap at `max_depth=None`:** training accuracy is 1.000 but CV accuracy is 0.724. The tree has 35 leaves for 207 patients, so it has memorised the training rows (including their noise), and the 0.28 gap is the over-fitting: training accuracy says nothing about new patients, cross-validated accuracy does.

# Part 4. Tree or logistic regression?

## 4a. Cross-validation on the same folds (`cv`)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

models = {
    f"tree (max_depth={chosen_depth})": DecisionTreeClassifier(criterion="entropy", max_depth=chosen_depth, random_state=SEED),
    "tree (unlimited)": DecisionTreeClassifier(criterion="entropy", max_depth=None, random_state=SEED),
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),   # scaler fitted inside each training fold
}

rows, fold_acc = [], {}
for name, m in models.items():
    s = cross_validate(m, X_train, y_train, cv=cv, scoring=("accuracy", "f1"))
    fold_acc[name] = s["test_accuracy"]
    rows.append({"model": name,
                 "cv_acc_mean": s["test_accuracy"].mean(), "cv_acc_std": s["test_accuracy"].std(),
                 "cv_f1_mean": s["test_f1"].mean()})
cv_table = pd.DataFrame(rows)
cv_table.round(4)

In [ ]:
# Same folds for every model, so the per-fold differences are paired
fold_df = pd.DataFrame(fold_acc).round(4)
fold_df.index = [f"fold {i + 1}" for i in range(len(fold_df))]
print(fold_df, "\n")

for other in [c for c in fold_df.columns if c != "logistic regression"]:
    d = fold_df["logistic regression"] - fold_df[other]
    print(f"logreg minus {other}: mean {d.mean():+.4f}, std {d.std(ddof=1):.4f}, logreg better in {(d > 0).sum()} of {len(d)} folds")

## 4b. Does a tree need scaling?

In [ ]:
pred_raw = DecisionTreeClassifier(criterion="entropy", max_depth=chosen_depth, random_state=SEED) \
    .fit(X_train, y_train).predict(X_test)

scaler = StandardScaler().fit(X_train)
pred_scaled = DecisionTreeClassifier(criterion="entropy", max_depth=chosen_depth, random_state=SEED) \
    .fit(scaler.transform(X_train), y_train).predict(scaler.transform(X_test))

print("fraction of identical test predictions:", (pred_raw == pred_scaled).mean())

**Explanation.** A tree only ever asks "is $x_j \le t$?", and `StandardScaler` replaces each feature by $(x_j-\mu_j)/\sigma_j$ with $\sigma_j>0$: an increasing transformation that does not change the order of the values within a feature.
So the same partitions of the training rows are candidates, with the same information gains; the midpoint thresholds are mapped to the new midpoints, and every test row falls on the same side as before.
The fitted tree is the same tree in different units, hence the agreement of 1.0 (it could be below 1 only through a floating-point rounding in a tie). Logistic regression is different: its coefficients and its regularisation depend on the scale of the features, so it needs the scaler.

## 4c. Hold-out, once

In [ ]:
tree_final = DecisionTreeClassifier(criterion="entropy", max_depth=chosen_depth, random_state=SEED).fit(X_train, y_train)
logreg_final = models["logistic regression"].fit(X_train, y_train)

n = len(y_test)
for name, m in [(f"tree (max_depth={chosen_depth})", tree_final), ("logistic regression", logreg_final)]:
    pred = m.predict(X_test)
    acc = accuracy_score(y_test, pred)
    se = np.sqrt(acc * (1 - acc) / n)
    print(f"{name}: accuracy = {acc:.4f} (SE {se:.4f}), F1 = {f1_score(y_test, pred):.4f}")
    print("confusion matrix (rows = true 0/1, columns = predicted 0/1):")
    print(confusion_matrix(y_test, pred), "\n")

## 4d. Conclusion

I would use **logistic regression**, with moderate confidence. In cross-validation on the training split it reaches an accuracy of 0.845 ± 0.065 and F1 of 0.826, against 0.724 ± 0.075 (F1 0.677) for the unlimited tree and 0.686 ± 0.078 (F1 0.660) for the depth-1 tree: a gap of 0.12–0.16 that is larger than the fold std of any of the models (the per-fold table above shows how often it wins on the same folds). On the 90 hold-out rows the two models are indistinguishable: accuracy 0.800 (SE 0.042) for logistic regression and 0.789 (SE 0.043) for the tree, F1 0.775 against 0.782, a difference of one patient, far inside the noise. I also cannot tell apart the unlimited tree from the depth-1 tree in CV (0.04 against a fold std of about 0.075), nor the different depths in the Part 3 sweep. I trust the CV comparison more than the hold-out, since it rests on 207 predictions per model instead of 90 and all the folds point in the same direction; the hold-out neither confirms nor contradicts it. Both rest on a single split with seed 421, and this is a teaching dataset, so none of it is a medical conclusion.

# Part 5. Reading a tree, and reproducibility

In [ ]:
from sklearn.tree import export_text

shallow = DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=SEED).fit(X_train, y_train)
print(export_text(shallow, feature_names=feature_names))

ranked = sorted(zip(shallow.feature_importances_, feature_names), reverse=True)
print([(name, round(float(imp), 3)) for imp, name in ranked[:5]])

In [ ]:
# Leaves of the depth-3 tree: how many training patients reach each one, and how they split by class
t = shallow.tree_
rows = []
for i in range(t.node_count):
    if t.children_left[i] == -1:                                   # a leaf
        v = t.value[i][0]
        counts = np.round(v / v.sum() * t.n_node_samples[i]).astype(int)   # works whether value holds counts or fractions
        rows.append({"leaf": i, "patients": int(t.n_node_samples[i]),
                     "no disease (0)": int(counts[0]), "disease (1)": int(counts[1]),
                     "predicts": int(shallow.classes_[v.argmax()])})
leaves_df = pd.DataFrame(rows)
leaves_df

## 5.1 Two rules in plain English

Leaf numbers refer to the table above (counts are training patients; the class split of each leaf is in the table).

- **Predicts disease (leaf 13).** If the thallium result is a fixed or reversible defect (`thal` > 4.5, i.e. 6 or 7), **and** the ST depression after exercise is above 0.7 (`oldpeak` > 0.7), **and** the maximum heart rate is at most 175.5 bpm (`thalach` ≤ 175.5), then predict **disease**. **54** training patients reach this leaf.
- **Predicts no disease (leaf 4).** If the thallium result is normal (`thal` ≤ 4.5, i.e. 3), **and** no major vessel is coloured by fluoroscopy (`ca` ≤ 0.5, i.e. 0), **and** the maximum heart rate is above 160.5 bpm (`thalach` > 160.5), then predict **no disease**. **45** training patients reach this leaf.

## 5.2 Importances

The five most important features of my depth-3 tree: **`thal` (0.379), `thalach` (0.179), `ca` (0.161), `oldpeak` (0.102), `cp` (0.096)**.

*Limitation of `feature_importances_`:* it is the total impurity decrease that the splits on a feature produce **on the training data**, so it describes this one fitted tree, not how much the feature matters in the world. It is biased towards continuous or many-valued features (more candidate thresholds, more chances to find a good-looking split on noise), it gets split arbitrarily between correlated features, and it says nothing about the direction of the effect. It also cannot be compared across different trees.

In [ ]:
# The pair of leaves that predict the same class: the two children of the thalach split (leaves 3 and 4)
parent = [i for i in range(t.node_count) if t.children_left[i] != -1 and t.children_left[i] == 3][0]
l, r = t.children_left[parent], t.children_right[parent]
print("split:", feature_names[t.feature[parent]], "<=", t.threshold[parent])
for name, node in [("left", l), ("right", r)]:
    row = leaves_df[leaves_df["leaf"] == node].iloc[0]
    print(f"{name} child (leaf {node}): {row['patients']} patients, {row['disease (1)']} with disease "
          f"({row['disease (1)'] / row['patients']:.1%}) -> predicts {row['predicts']}")

y_node = [y_train[shallow.apply(X_train) == node] for node in (l, r)]
y_par = np.concatenate(y_node)
print("entropy of the parent node:", round(entropy(y_par), 4),
      "| weighted entropy of the two children:",
      round(sum(len(a) / len(y_par) * entropy(a) for a in y_node), 4))

## 5.3 Why a split can have two children that predict the same class

Leaves 3 and 4 (the split `thalach <= 160.5` under `thal <= 4.5` and `ca <= 0.5`) both predict "no disease", yet the split exists: scikit-learn does not optimise accuracy, it minimises the **weighted entropy of the children** (maximises information gain, since I asked for `criterion="entropy"`).
Entropy is strictly concave, so splitting a node into two children whose proportions of disease differ (see the percentages above) lowers the weighted entropy even though the majority class is "no disease" on both sides; the accuracy does not change, but the children are purer, which is what later splits and `predict_proba` use.

## Reproducibility

The first cell prints the library versions, SEED = 421 is written in the markdown cell under it, and the notebook is meant to run from top to bottom after *Restart & Run All*.